In [1]:
import pandas as pd
from pathlib import Path

In [2]:
processed_path = Path("../data/processed")

glucose = pd.read_csv(
    processed_path / "glucose_level.csv",
    parse_dates=["ts"]
)

meal = pd.read_csv(
    processed_path / "meal.csv",
    parse_dates=["ts"]
)

bolus = pd.read_csv(
    processed_path / "bolus.csv",
    parse_dates=["ts_begin"]
)

basal = pd.read_csv(
    processed_path / "basal.csv",
    parse_dates=["ts"]
)

temp_basal = pd.read_csv(
    processed_path / "temp_basal.csv",
    parse_dates=["ts_begin"]
)

exercise = pd.read_csv(
    processed_path / "exercise.csv",
    parse_dates=["ts"]
)

In [3]:
print(glucose.shape)
print(meal.shape)
print(bolus.shape)

(65535, 4)
(702, 5)
(1568, 6)


In [4]:
glucose = glucose.rename(columns = {
    'ts': 'timestamp',
    'value': 'glucose_value'
})

meal = meal.rename(columns = {
    'ts': 'timestamp',
    'carbs': 'meal_carbs',
    'type': 'meal_type'
})

bolus = bolus.rename(columns = {
    'ts_begin': 'timestamp',
    'dose': 'bolus_dose',
    'type': 'bolus_type'
})

basal = basal.rename(columns = {
    'ts': 'timestamp',
    'value': 'basal_value'
})

temp_basal = temp_basal.rename(columns = {
    'ts_begin': 'timestamp',
    'value': 'temp_basal_value'
})

exercise = exercise.rename(columns = {
    'ts': 'timestamp',
    'type': 'exercise_type',
    'duration': 'exercise_duration',
    'intensity': 'exercise_intensity'
})

In [5]:
datasets = [
    glucose,
    meal,
    bolus,
    basal,
    temp_basal,
    exercise
]

for df in datasets:
    df.sort_values(
        ['Patient', 'timestamp'],
        inplace=True
    )

In [6]:
merged = glucose.copy()

merged.head()

,Patient,Section,timestamp,glucose_value
43635,540-ws-training,glucose_level,2027-05-19 11:36:29,76
43638,540-ws-training,glucose_level,2027-05-19 11:41:29,72
43641,540-ws-training,glucose_level,2027-05-19 11:46:29,68
43644,540-ws-training,glucose_level,2027-05-19 11:51:29,65
43647,540-ws-training,glucose_level,2027-05-19 11:56:29,63


In [7]:
merged = pd.merge_asof(
    merged.sort_values("timestamp"),
    meal.sort_values("timestamp"),
    on="timestamp",
    by="Patient",
    direction="backward"
)

merged.head()

,Patient,Section_x,timestamp,glucose_value,Section_y,meal_type,meal_carbs
0,552-ws-training,glucose_level,2025-04-16 11:17:05,95,NaN,NaN,NaN
1,552-ws-training,glucose_level,2025-04-16 11:22:05,86,NaN,NaN,NaN
2,552-ws-training,glucose_level,2025-04-16 11:27:05,81,NaN,NaN,NaN
3,552-ws-training,glucose_level,2025-04-16 11:32:05,81,NaN,NaN,NaN
4,552-ws-training,glucose_level,2025-04-16 11:37:05,82,NaN,NaN,NaN


In [8]:
print(merged.shape)
print(merged.columns)

(65535, 7)
Index(['Patient', 'Section_x', 'timestamp', 'glucose_value', 'Section_y',
       'meal_type', 'meal_carbs'],
      dtype='object')


In [9]:
print(merged["meal_carbs"].notna().sum())

64026


In [10]:
merged[merged["meal_carbs"].notna()].head(10)

,Patient,Section_x,timestamp,glucose_value,Section_y,meal_type,meal_carbs
81,552-ws-training,glucose_level,2025-04-16 18:02:06,228,meal,Dinner,30.0
82,552-ws-training,glucose_level,2025-04-16 18:07:06,232,meal,Dinner,30.0
83,552-ws-training,glucose_level,2025-04-16 18:12:06,234,meal,Dinner,30.0
84,552-ws-training,glucose_level,2025-04-16 18:17:06,233,meal,Dinner,30.0
85,552-ws-training,glucose_level,2025-04-16 18:22:06,234,meal,Dinner,30.0
86,552-ws-training,glucose_level,2025-04-16 18:27:06,233,meal,Dinner,30.0
87,552-ws-training,glucose_level,2025-04-16 18:32:06,230,meal,Dinner,30.0
88,552-ws-training,glucose_level,2025-04-16 18:37:06,235,meal,Dinner,30.0
89,552-ws-training,glucose_level,2025-04-16 18:42:06,242,meal,Dinner,30.0
90,552-ws-training,glucose_level,2025-04-16 18:47:06,241,meal,Dinner,30.0


In [11]:
merged = merged.drop(
    columns=["Section_x", "Section_y"],
    errors="ignore"
)

In [12]:
print(merged.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs']


In [13]:
print(bolus.columns.tolist())

['Patient', 'Section', 'timestamp', 'ts_end', 'bolus_type', 'bolus_dose']


In [14]:
bolus = bolus[[
    "Patient",
    "timestamp",
    "bolus_type",
    "bolus_dose"
]].copy()

In [15]:
# def merge_patient_asof(left_df, right_df):

#     merged_list = []

#     for patient in sorted(left_df["Patient"].unique()):

#         left = (
#             left_df[left_df["Patient"] == patient]
#             .sort_values("timestamp")
#             .reset_index(drop=True)
#         )

#         right = (
#             right_df[right_df["Patient"] == patient]
#             .sort_values("timestamp")
#             .reset_index(drop=True)
#         )

#         temp = pd.merge_asof(
#             left,
#             right,
#             on="timestamp",
#             direction="backward"
#         )

#         merged_list.append(temp)

#     return pd.concat(merged_list, ignore_index=True)

In [16]:
def merge_patient_asof(left_df, right_df):

    merged_list = []

    for patient in sorted(left_df["Patient"].unique()):

        left = (
            left_df[left_df["Patient"] == patient]
            .sort_values("timestamp")
            .reset_index(drop=True)
        )

        right = (
            right_df[right_df["Patient"] == patient]
            .sort_values("timestamp")
            .reset_index(drop=True)
        )

        # Remove Patient column from the right dataframe
        right = right.drop(columns=["Patient"], errors="ignore")

        temp = pd.merge_asof(
            left,
            right,
            on="timestamp",
            direction="backward"
        )

        merged_list.append(temp)

    return pd.concat(merged_list, ignore_index=True)

In [17]:
merged = merge_patient_asof(merged, bolus)

In [18]:
print(merged.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose']


In [19]:
def merge_patient_asof(left_df, right_df):

    merged_list = []

    for patient in sorted(left_df["Patient"].unique()):

        left = (
            left_df[left_df["Patient"] == patient]
            .sort_values("timestamp")
            .reset_index(drop=True)
        )

        right = (
            right_df[right_df["Patient"] == patient]
            .sort_values("timestamp")
            .reset_index(drop=True)
        )

        # Remove Patient column from the right dataframe
        right = right.drop(columns=["Patient"], errors="ignore")

        temp = pd.merge_asof(
            left,
            right,
            on="timestamp",
            direction="backward"
        )

        merged_list.append(temp)

    return pd.concat(merged_list, ignore_index=True)

In [20]:
print(basal.columns.tolist())

['Patient', 'Section', 'timestamp', 'basal_value']


In [21]:
basal = basal[[
    'Patient',
    'timestamp',
    'basal_value'
]].copy()

In [22]:
print(basal.head())
print(basal.columns.tolist())

             Patient           timestamp  basal_value
228  540-ws-training 2027-05-19 00:00:00         0.80
229  540-ws-training 2027-05-19 05:00:00         1.05
230  540-ws-training 2027-05-19 09:00:00         0.95
231  540-ws-training 2027-05-19 14:00:00         0.40
232  540-ws-training 2027-05-22 00:00:00         0.80
['Patient', 'timestamp', 'basal_value']


In [23]:
merged = merge_patient_asof(merged, basal)

In [24]:
if "Patient_x" in merged.columns:
    merged = merged.rename(columns={"Patient_x": "Patient"})

if "Patient_y" in merged.columns:
    merged = merged.drop(columns=["Patient_y"])

In [25]:
print(merged.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value']


In [26]:
merged.head(10)

,Patient,timestamp,glucose_value,meal_type,meal_carbs,bolus_type,bolus_dose,basal_value
0,540-ws-training,2027-05-19 11:36:29,76,NaN,NaN,normal,0.8,0.95
1,540-ws-training,2027-05-19 11:41:29,72,NaN,NaN,normal,0.8,0.95
2,540-ws-training,2027-05-19 11:46:29,68,NaN,NaN,normal,0.8,0.95
3,540-ws-training,2027-05-19 11:51:29,65,NaN,NaN,normal,0.8,0.95
4,540-ws-training,2027-05-19 11:56:29,63,NaN,NaN,normal,0.8,0.95
5,540-ws-training,2027-05-19 12:01:29,66,NaN,NaN,normal,0.8,0.95
6,540-ws-training,2027-05-19 12:06:29,71,NaN,NaN,normal,0.8,0.95
7,540-ws-training,2027-05-19 12:11:29,78,NaN,NaN,normal dual,5.5,0.95
8,540-ws-training,2027-05-19 12:16:29,90,NaN,NaN,normal dual,5.5,0.95
9,540-ws-training,2027-05-19 12:21:29,99,NaN,NaN,normal dual,5.5,0.95


In [27]:
print(merged.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value']


In [28]:
temp_basal = temp_basal[[
    'Patient',
    'timestamp',
    'temp_basal_value'
]].copy()

In [29]:
print(temp_basal.columns.tolist())

['Patient', 'timestamp', 'temp_basal_value']


In [30]:
merged = merge_patient_asof(merged, temp_basal)

In [31]:
if "Patient_x" in merged.columns:
    merged = merged.rename(columns={"Patient_x": "Patient"})

if "Patient_y" in merged.columns:
    merged = merged.drop(columns=["Patient_y"])

In [32]:
print(merged.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value', 'temp_basal_value']


In [33]:
exercise = exercise[[
    'Patient',
    'timestamp',
    'exercise_type',
    'exercise_duration',
    'exercise_intensity'
]].copy()

In [34]:
merged = merge_patient_asof(merged, exercise)

In [35]:
print(merged.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value', 'temp_basal_value', 'exercise_type', 'exercise_duration', 'exercise_intensity']


In [36]:
finger_stick_path = processed_path / "finger_stick.csv"

if not finger_stick_path.exists():
    raise FileNotFoundError(f"{finger_stick_path} not found. Load the finger_stick dataset first.")

finger_stick = pd.read_csv(
    finger_stick_path,
    parse_dates=["ts"]
).rename(columns={"ts": "timestamp"})

finger_stick = finger_stick[[
    "Patient",
    "timestamp",
    "value"
]].copy()

In [37]:
merged = merge_patient_asof(merged, finger_stick)

In [38]:
print(merged.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value', 'temp_basal_value', 'exercise_type', 'exercise_duration', 'exercise_intensity', 'value']


In [39]:
print([x for x in locals().keys() if "gsr" in x.lower()])

[]


In [40]:
basis_gsr = pd.read_csv(
    processed_path / "basis_gsr.csv"
)

In [41]:
print(basis_gsr.columns.tolist())
print(basis_gsr.head())

['Patient', 'Section', 'ts', 'value']
           Patient    Section                   ts     value
0  552-ws-training  basis_gsr  2025-04-17 20:01:00  0.013224
1  552-ws-training  basis_gsr  2025-04-17 20:02:00  0.014324
2  552-ws-training  basis_gsr  2025-04-17 20:03:00  0.016366
3  552-ws-training  basis_gsr  2025-04-17 20:04:00  0.018619
4  552-ws-training  basis_gsr  2025-04-17 20:05:00  0.019737


In [42]:
basis_gsr = basis_gsr.rename(columns={
    "ts": "timestamp",
    "value": "gsr"
})

In [43]:
basis_gsr = basis_gsr[[
    "Patient",
    "timestamp",
    "gsr"
]].copy()

In [44]:
print("merged:", merged["timestamp"].dtype)
print("basis_gsr:", basis_gsr["timestamp"].dtype)

merged: datetime64[ns]
basis_gsr: object


In [45]:
merged["timestamp"] = pd.to_datetime(
    merged["timestamp"],
    errors="coerce"
)

basis_gsr["timestamp"] = pd.to_datetime(
    basis_gsr["timestamp"],
    errors="coerce"
)

In [46]:
print("merged:", merged["timestamp"].dtype)
print("basis_gsr:", basis_gsr["timestamp"].dtype)

merged: datetime64[ns]
basis_gsr: datetime64[ns]


In [47]:
merged = merged.dropna(subset=["timestamp"]).copy()
basis_gsr = basis_gsr.dropna(subset=["timestamp"]).copy()

In [48]:
merged = merge_patient_asof(merged, basis_gsr)

In [49]:
basis_skin_temperature = pd.read_csv(
    processed_path / "basis_skin_temperature.csv"
)

basis_skin_temperature = basis_skin_temperature.rename(columns={
    "ts": "timestamp"
})

basis_skin_temperature["timestamp"] = pd.to_datetime(
    basis_skin_temperature["timestamp"],
    errors="coerce"
)

basis_skin_temperature = basis_skin_temperature.dropna(
    subset=["timestamp"]
)

basis_skin_temperature = basis_skin_temperature[[
    "Patient",
    "timestamp",
    "value"
]].copy()

In [50]:
merged = merge_patient_asof(
    merged,
    basis_skin_temperature
)

In [51]:
print(merged.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value', 'temp_basal_value', 'exercise_type', 'exercise_duration', 'exercise_intensity', 'value_x', 'gsr', 'value_y']


In [52]:
acceleration = pd.read_csv(
    processed_path / "acceleration.csv"
)

acceleration = acceleration.rename(columns={
    "ts": "timestamp"
})

acceleration["timestamp"] = pd.to_datetime(
    acceleration["timestamp"],
    errors="coerce"
)

acceleration = acceleration.dropna(
    subset=["timestamp"]
)

acceleration = acceleration[[
    "Patient",
    "timestamp",
    "value"
]].copy()

In [53]:
merged = merge_patient_asof(
    merged,
    acceleration
)

In [54]:
print(merged.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value', 'temp_basal_value', 'exercise_type', 'exercise_duration', 'exercise_intensity', 'value_x', 'gsr', 'value_y', 'value']


In [55]:
merged = merged.rename(columns={
    "value_x": "fingerstick_glucose",
    "value_y": "skin_temperature",
    "value": "acceleration"
})

In [56]:
print(merged.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value', 'temp_basal_value', 'exercise_type', 'exercise_duration', 'exercise_intensity', 'fingerstick_glucose', 'gsr', 'skin_temperature', 'acceleration']


In [57]:
print(finger_stick.columns.tolist())
print(basis_skin_temperature.columns.tolist())
print(acceleration.columns.tolist())

['Patient', 'timestamp', 'value']
['Patient', 'timestamp', 'value']
['Patient', 'timestamp', 'value']


In [58]:
basis_sleep = pd.read_csv(
    processed_path / "basis_sleep.csv"
)

In [59]:
print(basis_sleep.columns.tolist())

['Patient', 'Section', 'tbegin', 'tend', 'quality', 'type']


In [60]:
basis_sleep = basis_sleep.rename(columns={
    "tbegin": "start_time",
    "tend": "end_time"
})

In [61]:
basis_sleep["start_time"] = pd.to_datetime(
    basis_sleep["start_time"],
    errors="coerce"
)

basis_sleep["end_time"] = pd.to_datetime(
    basis_sleep["end_time"],
    errors="coerce"
)


In [62]:
merged["is_sleeping"] = 0

In [63]:
for patient in merged["Patient"].unique():

    sleep_patient = basis_sleep[
        basis_sleep["Patient"] == patient
    ]

    for _, row in sleep_patient.iterrows():

        mask = (
            (merged["Patient"] == patient) &
            (merged["timestamp"] >= row["start_time"]) &
            (merged["timestamp"] <= row["end_time"])
        )

        merged.loc[mask, "is_sleeping"] = 1

In [64]:
print(merged["is_sleeping"].value_counts())

is_sleeping
0    54654
1    10881
Name: count, dtype: int64


In [65]:
print(merged["is_sleeping"].value_counts())

is_sleeping
0    54654
1    10881
Name: count, dtype: int64


In [66]:
sleep = pd.read_csv(
    processed_path / "sleep.csv"
)

In [67]:
print(sleep.columns.tolist())

['Patient', 'Section', 'ts_begin', 'ts_end', 'quality']


In [68]:
sleep = sleep.rename(columns={
    "ts_begin": "start_time",
    "ts_end": "end_time"
})

sleep["start_time"] = pd.to_datetime(
    sleep["start_time"],
    errors="coerce"
)

sleep["end_time"] = pd.to_datetime(
    sleep["end_time"],
    errors="coerce"
)

In [69]:
merged["sleep_quality"] = pd.NA

In [70]:
for patient in merged["Patient"].unique():

    sleep_patient = sleep[
        sleep["Patient"] == patient
    ]

    for _, row in sleep_patient.iterrows():

        mask = (
            (merged["Patient"] == patient) &
            (merged["timestamp"] >= row["start_time"]) &
            (merged["timestamp"] <= row["end_time"])
        )

        merged.loc[mask, "sleep_quality"] = row["quality"]

In [71]:
print(merged["sleep_quality"].value_counts(dropna=False))

sleep_quality
<NA>    65535
Name: count, dtype: int64


In [72]:
print(sleep.head(2))

           Patient Section          start_time            end_time  quality
0  552-ws-training   sleep 2025-04-17 05:36:00 2025-04-16 23:00:00        2
1  552-ws-training   sleep 2025-04-23 06:00:00 2025-04-22 22:32:00        3


In [73]:
print(sleep[["start_time", "end_time", "quality"]].head(10))

           start_time            end_time  quality
0 2025-04-17 05:36:00 2025-04-16 23:00:00        2
1 2025-04-23 06:00:00 2025-04-22 22:32:00        3
2 2025-04-24 05:35:00 2025-04-23 23:00:00        2
3 2025-04-25 09:32:00 2025-04-25 00:30:00        3
4 2025-04-26 09:39:00 2025-04-25 22:39:00        3
5 2025-04-27 09:00:00 2025-04-26 23:28:00        3
6 2025-04-29 06:31:00 2025-04-28 23:31:00        1
7 2025-04-30 05:30:00 2025-04-29 22:30:00        2
8 2025-05-01 05:35:00 2025-04-30 23:15:00        3
9 2025-05-06 05:40:00 2025-05-05 22:52:00        3


In [74]:
print(sleep["start_time"].min())
print(sleep["start_time"].max())

print(merged["timestamp"].min())
print(merged["timestamp"].max())

2025-04-17 05:36:00
2027-06-23 06:00:00
2025-04-16 11:17:05
2027-07-03 23:56:44


In [75]:
print(sleep["quality"].value_counts(dropna=False))

quality
3    83
2    53
1    14
Name: count, dtype: int64


In [76]:
print("Sleep:")
print("Start:", sleep["start_time"].min())
print("End:", sleep["end_time"].max())

Sleep:
Start: 2025-04-17 05:36:00
End: 2027-06-22 21:30:00


In [77]:
print("Merged:")
print("Start:", merged["timestamp"].min())
print("End:", merged["timestamp"].max())

Merged:
Start: 2025-04-16 11:17:05
End: 2027-07-03 23:56:44


In [78]:
print("Sleep patients:")
print(sleep["Patient"].unique())

print("\nMerged patients:")
print(merged["Patient"].unique())

Sleep patients:
['552-ws-training' '584-ws-training' '567-ws-training' '596-ws-training'
 '544-ws-training']

Merged patients:
['540-ws-training' '544-ws-training' '552-ws-training' '567-ws-training'
 '584-ws-training' '596-ws-training']


In [79]:
print(sleep[[
    "Patient",
    "start_time",
    "end_time",
    "quality"
]].head(10))

           Patient          start_time            end_time  quality
0  552-ws-training 2025-04-17 05:36:00 2025-04-16 23:00:00        2
1  552-ws-training 2025-04-23 06:00:00 2025-04-22 22:32:00        3
2  552-ws-training 2025-04-24 05:35:00 2025-04-23 23:00:00        2
3  552-ws-training 2025-04-25 09:32:00 2025-04-25 00:30:00        3
4  552-ws-training 2025-04-26 09:39:00 2025-04-25 22:39:00        3
5  552-ws-training 2025-04-27 09:00:00 2025-04-26 23:28:00        3
6  552-ws-training 2025-04-29 06:31:00 2025-04-28 23:31:00        1
7  552-ws-training 2025-04-30 05:30:00 2025-04-29 22:30:00        2
8  552-ws-training 2025-05-01 05:35:00 2025-04-30 23:15:00        3
9  552-ws-training 2025-05-06 05:40:00 2025-05-05 22:52:00        3


In [80]:
patient = sleep["Patient"].iloc[0]

print("Patient:", patient)

print("\nSleep interval:")
print(sleep[sleep["Patient"] == patient][[
    "start_time",
    "end_time",
    "quality"
]].head())

print("\nGlucose records for same patient:")
print(merged[merged["Patient"] == patient][[
    "timestamp",
    "glucose_value"
]].head(20))

Patient: 552-ws-training

Sleep interval:
           start_time            end_time  quality
0 2025-04-17 05:36:00 2025-04-16 23:00:00        2
1 2025-04-23 06:00:00 2025-04-22 22:32:00        3
2 2025-04-24 05:35:00 2025-04-23 23:00:00        2
3 2025-04-25 09:32:00 2025-04-25 00:30:00        3
4 2025-04-26 09:39:00 2025-04-25 22:39:00        3

Glucose records for same patient:
                timestamp  glucose_value
22570 2025-04-16 11:17:05             95
22571 2025-04-16 11:22:05             86
22572 2025-04-16 11:27:05             81
22573 2025-04-16 11:32:05             81
22574 2025-04-16 11:37:05             82
22575 2025-04-16 11:42:05             82
22576 2025-04-16 11:47:05             84
22577 2025-04-16 11:52:05             88
22578 2025-04-16 11:57:05             98
22579 2025-04-16 12:02:05            110
22580 2025-04-16 12:07:05             96
22581 2025-04-16 12:12:05             97
22582 2025-04-16 12:17:05             97
22583 2025-04-16 12:22:05             95
22

In [81]:
print(
    merged["timestamp"].dtype,
    sleep["start_time"].dtype,
    sleep["end_time"].dtype
)

datetime64[ns] datetime64[ns] datetime64[ns]


In [82]:
# Pick the first sleep record
row = sleep.iloc[0]

print("SLEEP RECORD")
print("Patient:", row["Patient"])
print("Start:", row["start_time"])
print("End:", row["end_time"])
print("Quality:", row["quality"])

# Find glucose records for the same patient
test = merged[
    (merged["Patient"] == row["Patient"]) &
    (merged["timestamp"] >= row["start_time"]) &
    (merged["timestamp"] <= row["end_time"])
]

print("\nMATCHING GLUCOSE RECORDS:")
print(test[["Patient", "timestamp", "glucose_value"]].head(10))

print("\nNumber of matches:", len(test))


SLEEP RECORD
Patient: 552-ws-training
Start: 2025-04-17 05:36:00
End: 2025-04-16 23:00:00
Quality: 2

MATCHING GLUCOSE RECORDS:
Empty DataFrame
Columns: [Patient, timestamp, glucose_value]
Index: []

Number of matches: 0


In [83]:
# Fix overnight/reversed sleep intervals
mask = sleep["end_time"] < sleep["start_time"]

print("Reversed intervals:", mask.sum())

sleep.loc[mask, "start_time"] = (
    sleep.loc[mask, "start_time"] - pd.Timedelta(days=1)
)

Reversed intervals: 150


In [84]:
print(sleep[["Patient", "start_time", "end_time", "quality"]].head(10))

           Patient          start_time            end_time  quality
0  552-ws-training 2025-04-16 05:36:00 2025-04-16 23:00:00        2
1  552-ws-training 2025-04-22 06:00:00 2025-04-22 22:32:00        3
2  552-ws-training 2025-04-23 05:35:00 2025-04-23 23:00:00        2
3  552-ws-training 2025-04-24 09:32:00 2025-04-25 00:30:00        3
4  552-ws-training 2025-04-25 09:39:00 2025-04-25 22:39:00        3
5  552-ws-training 2025-04-26 09:00:00 2025-04-26 23:28:00        3
6  552-ws-training 2025-04-28 06:31:00 2025-04-28 23:31:00        1
7  552-ws-training 2025-04-29 05:30:00 2025-04-29 22:30:00        2
8  552-ws-training 2025-04-30 05:35:00 2025-04-30 23:15:00        3
9  552-ws-training 2025-05-05 05:40:00 2025-05-05 22:52:00        3


In [85]:
mask = sleep["end_time"] < sleep["start_time"]

sleep.loc[mask, ["start_time", "end_time"]] = (
    sleep.loc[mask, ["end_time", "start_time"]].to_numpy()
)

In [86]:
mask = sleep["end_time"] < sleep["start_time"]

sleep.loc[mask, ["start_time", "end_time"]] = (
    sleep.loc[mask, ["end_time", "start_time"]].to_numpy()
)

In [87]:
print(sleep[["Patient", "start_time", "end_time", "quality"]].head(10))

print(
    "Still reversed:",
    (sleep["end_time"] < sleep["start_time"]).sum()
)

           Patient          start_time            end_time  quality
0  552-ws-training 2025-04-16 05:36:00 2025-04-16 23:00:00        2
1  552-ws-training 2025-04-22 06:00:00 2025-04-22 22:32:00        3
2  552-ws-training 2025-04-23 05:35:00 2025-04-23 23:00:00        2
3  552-ws-training 2025-04-24 09:32:00 2025-04-25 00:30:00        3
4  552-ws-training 2025-04-25 09:39:00 2025-04-25 22:39:00        3
5  552-ws-training 2025-04-26 09:00:00 2025-04-26 23:28:00        3
6  552-ws-training 2025-04-28 06:31:00 2025-04-28 23:31:00        1
7  552-ws-training 2025-04-29 05:30:00 2025-04-29 22:30:00        2
8  552-ws-training 2025-04-30 05:35:00 2025-04-30 23:15:00        3
9  552-ws-training 2025-05-05 05:40:00 2025-05-05 22:52:00        3
Still reversed: 0


In [88]:
merged["sleep_quality"] = pd.NA

In [89]:
for patient in merged["Patient"].unique():

    sleep_patient = sleep[
        sleep["Patient"] == patient
    ]

    for _, row in sleep_patient.iterrows():

        mask = (
            (merged["Patient"] == patient) &
            (merged["timestamp"] >= row["start_time"]) &
            (merged["timestamp"] <= row["end_time"])
        )

        merged.loc[mask, "sleep_quality"] = row["quality"]

In [90]:
print(merged["sleep_quality"].value_counts(dropna=False))

sleep_quality
<NA>    41554
3       13769
2        7947
1        2265
Name: count, dtype: int64


In [94]:
illness = pd.read_csv(
    processed_path / "illness.csv"
)

In [95]:
print(illness)

           Patient  Section             ts_begin  ts_end  type description
0  552-ws-training  illness  2025-04-16 20:11:00     NaN   NaN            
1  567-ws-training  illness  2027-01-18 12:26:00     NaN   NaN            
2  596-ws-training  illness  2027-05-12 03:25:00     NaN   NaN            


In [96]:
print(illness[["Patient", "ts_begin", "ts_end", "type", "description"]])

           Patient             ts_begin  ts_end  type description
0  552-ws-training  2025-04-16 20:11:00     NaN   NaN            
1  567-ws-training  2027-01-18 12:26:00     NaN   NaN            
2  596-ws-training  2027-05-12 03:25:00     NaN   NaN            


In [98]:
work = pd.read_csv(
    processed_path / "work.csv"
)

In [99]:
print(work.columns.tolist())

['Patient', 'Section', 'ts_begin', 'ts_end', 'intensity']


In [100]:
work = work.rename(columns={
    "ts_begin": "start_time",
    "ts_end": "end_time",
    "intensity": "work_intensity"
})

work["start_time"] = pd.to_datetime(
    work["start_time"],
    errors="coerce"
)

work["end_time"] = pd.to_datetime(
    work["end_time"],
    errors="coerce"
)

In [101]:
merged["is_working"] = 0

In [102]:
for patient in merged["Patient"].unique():

    work_patient = work[
        work["Patient"] == patient
    ]

    for _, row in work_patient.iterrows():

        mask = (
            (merged["Patient"] == patient) &
            (merged["timestamp"] >= row["start_time"]) &
            (merged["timestamp"] <= row["end_time"])
        )

        merged.loc[mask, "is_working"] = 1

In [103]:
merged["work_intensity"] = pd.NA

In [104]:
for patient in merged["Patient"].unique():

    work_patient = work[
        work["Patient"] == patient
    ]

    for _, row in work_patient.iterrows():

        mask = (
            (merged["Patient"] == patient) &
            (merged["timestamp"] >= row["start_time"]) &
            (merged["timestamp"] <= row["end_time"])
        )

        merged.loc[mask, "work_intensity"] = row["work_intensity"]

In [105]:
print(merged["is_working"].value_counts())
print()
print(merged["work_intensity"].value_counts(dropna=False))

is_working
0    60817
1     4718
Name: count, dtype: int64

work_intensity
<NA>    60817
5        1833
3        1496
2         696
4         587
6         106
Name: count, dtype: int64


In [106]:
stressor = pd.read_csv(
    processed_path / "stressor.csv"
)

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\processed\\stressor.csv'